# TxGNN — Step 1: heterogeneous training graph and frozen node inputs

**Execution plan and hands-on recipe • 20 November 2026**

Project target: an independently implemented, fully trainable TxGNN pipeline, checked against pinned public code, with ownership of preparation, training state, evaluation and exported models.

This notebook covers the preparation boundary only. The desired result is an immutable bundle that defines every node, relation, training edge, held-out view and input value used by later model code.

**Validation status:** the independent NumPy/pandas graph constructor and synthetic acceptance checks are exercised in this notebook. Full Dataverse acquisition, legacy-reference execution, PyTorch input generation and real-data parity have not been executed in the supplied environment. The notebook includes guarded code for those steps; it is not a claim that full-graph reproduction has already passed.

The included independent constructor starts from indexed, split tables. Implementing the raw-KG compatibility transform and disease split logic independently remains a work package below. The reference exporter supplies the oracle needed to develop and check that work, without importing the reference model into the independent constructor.

## 1. Freeze the specification

| Decision | Initial contract |
|---|---|
| Reference code | `mims-harvard/TxGNN`, commit `0e0203c3939f7d12da27aaf10e55daaff5424023` |
| Data | TxGNN loader's exact three Dataverse file IDs, plus pinned auxiliary files |
| Biological identity | Released harmonization and manually reviewed disease groups |
| Graph | Typed node inventory from the complete processed KG; adjacency from training rows |
| Inputs | One frozen CPU float32 Xavier-uniform matrix per type |
| Input width | 100 for the pinned reproduction script; 512 is a separately labelled paper configuration |
| Initial example | `complex_disease`, fold seed 42; change the explicit split for the actual experiment |
| Randomness | Separately record preprocessing seed, fold seed, input seed and actual realized state |
| Completion gate | Real-data graph parity, persisted-input parity and a fresh-process bundle reload |

The example split is an implementation smoke test, not a choice of the project's eventual disease-area evaluation. For `autoimmune`, retain the same output contract and apply the area-specific preparation below.

### What the attached paper establishes

Pages 22–23 report 10 node types, 29 undirected relation types, 123,527 nodes and 8,063,026 edges. They describe ontology harmonization, a largest connected component, DrugBank drug identifiers, MONDO diseases, and disease grouping using name matching, ClinicalBERT similarity ≥0.98, then manual approval and naming. The reported 22,205 MONDO concepts become 17,080 disease groups.

Use the released grouped graph. Rebuilding current versions of Bgee, CTD, DisGeNET, DrugBank, DrugCentral, Entrez Gene, GO, HPO, MONDO, interaction databases, Reactome, SIDER and UBERON is a separate curation project. ClinicalBERT in this passage concerns disease grouping; it is not the baseline's node-input encoder.

Published totals are reconciliation targets for a particular graph representation. Raw rows, representative facts, reverse-augmented rows and training adjacency need separate counts. Do not modify data merely to force a paper total.

## 2. Inputs to acquire

| Local filename | Dataverse file ID | Role | Delimiter |
|---|---:|---|---|
| `kg.csv` | 7144484 | Released KG, identities and relation rows | comma |
| `node.csv` | 7144482 | Global node indices and source identities | **tab** |
| `edges.csv` | 7144483 | Global-index edges used in area masking | comma |

The DOI is `10.7910/DVN/IXA7BM`, but the generic PrimeKG README points at a different `kg.csv` file ID. Follow the pinned TxGNN loader's file IDs for this reference target.

For disease-area splits, also preserve these exact repository files:

- `txgnn/data_splits/HumanDO.obo`
- `txgnn/data_splits/mondo_references.csv`
- `txgnn/data_splits/kg_grouped_diseases_bert_map.csv`
- `data/disease_files/<split>.csv`

The expected richer main-KG schema is `relation, display_relation, x_index, x_id, x_type, x_name, x_source, y_index, y_id, y_type, y_name, y_source`. Validate the actual acquired header. The downloader below checks the minimum fields read by the reference. It does not claim to have independently authenticated an unknown checksum: it records a first acquisition digest, which becomes the project's frozen input identity.

### Pin the public source in a separate directory

```bash
mkdir -p reference
git clone https://github.com/mims-harvard/TxGNN.git reference/TxGNN
git -C reference/TxGNN checkout --detach 0e0203c3939f7d12da27aaf10e55daaff5424023
git -C reference/TxGNN rev-parse HEAD
```

### Reference environment gate

The public README specifies Python 3.8 and DGL 0.5.2, with PyG needed for disease-area preprocessing. The repository does not supply a complete immutable dependency lock. Its use of `DataFrame.append` also makes an arbitrary current pandas installation unsuitable. Build and verify that isolated reference environment, then record all package versions and a `pip freeze`/container digest. Do not describe an untested package combination as a verified environment.

After installing compatible dependencies, install the checked-out source with `python -m pip install -e reference/TxGNN --no-deps`. The independent core below needs only Python, NumPy and pandas; it has no DGL or TxGNN import.


In [1]:
from pathlib import Path
import csv
import hashlib
import json
import os
import shutil
import sys
import tempfile
import urllib.request
from datetime import datetime, timezone
import numpy as np
import pandas as pd

PIN = '0e0203c3939f7d12da27aaf10e55daaff5424023'
WORK = Path.cwd() / 'txgnn_stage1_work'
PINNED_REPO = Path.cwd() / 'reference' / 'TxGNN'
CONFIG = {
    'reference_sha': PIN,
    'split': 'complex_disease',  # Set 'autoimmune' for that disease-area run.
    'fold_seed': 42,
    'preprocess_seed': 42,
    'input_seed': 0,
    'input_dim': 100,
    'no_kg': False,
    'test_size': 0.05,
    'one_hop': False,
    'mask_ratio': 0.1,
}

# Enable each expensive stage only after its prerequisites have been met.
RUN_DOWNLOAD = False
RUN_REFERENCE = False
RUN_REAL_COMPARISON = False
REFERENCE_SNAPSHOT = None  # Or set to an already exported snapshot directory.

ASSETS = {
    'kg.csv': (7144484, ',', {'x_index', 'y_index', 'x_type', 'x_id', 'relation', 'y_type', 'y_id'}),
    'node.csv': (7144482, '\t', {'node_index', 'node_id', 'node_source'}),
    'edges.csv': (7144483, ',', {'x_index', 'y_index', 'relation'}),
}

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def checked_header(path, delimiter, required):
    with Path(path).open('r', encoding='utf-8-sig', newline='') as stream:
        header = next(csv.reader(stream, delimiter=delimiter))
    missing = required.difference(header)
    if missing:
        raise ValueError('Invalid asset header: {} missing {}'.format(path, sorted(missing)))
    return header

def acquire_assets(raw_dir):
    raw_dir = Path(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    lock_path = raw_dir / 'asset_lock.json'
    prior = json.loads(lock_path.read_text()) if lock_path.exists() else {}
    records = {}
    for filename, (datafile_id, delimiter, required) in ASSETS.items():
        destination = raw_dir / filename
        url = 'https://dataverse.harvard.edu/api/access/datafile/{}'.format(datafile_id)
        if not destination.exists():
            descriptor, temporary = tempfile.mkstemp(dir=str(raw_dir), prefix='download_', suffix='.part')
            temporary = Path(temporary)
            try:
                with os.fdopen(descriptor, 'wb') as stream:
                    with urllib.request.urlopen(url, timeout=120) as response:
                        if response.status != 200:
                            raise RuntimeError('Unexpected HTTP status {}'.format(response.status))
                        shutil.copyfileobj(response, stream, length=1024 * 1024)
                checked_header(temporary, delimiter, required)
                if filename in prior and file_sha256(temporary) != prior[filename]['sha256']:
                    raise ValueError('Acquired bytes differ from the existing asset lock: ' + filename)
                os.replace(str(temporary), str(destination))
            finally:
                if temporary.exists():
                    temporary.unlink()
        header = checked_header(destination, delimiter, required)
        digest = file_sha256(destination)
        if filename in prior and digest != prior[filename]['sha256']:
            raise ValueError('Existing bytes differ from the asset lock: ' + filename)
        records[filename] = {
            'url': url, 'datafile_id': datafile_id, 'sha256': digest,
            'bytes': destination.stat().st_size, 'delimiter': delimiter, 'header': header,
            'verified_at_utc': datetime.now(timezone.utc).isoformat(),
        }
    temporary_lock = raw_dir / 'asset_lock.json.part'
    temporary_lock.write_text(json.dumps(records, indent=2) + '\n')
    os.replace(str(temporary_lock), str(lock_path))
    return records

if RUN_DOWNLOAD:
    ASSET_LOCK = acquire_assets(WORK / 'raw')
    print('Acquired and locally fingerprinted', len(ASSET_LOCK), 'assets.')
else:
    print('Acquisition recipe ready; full data download has not run.')


Acquisition recipe ready; full data download has not run.


## 3. Identity and schema work package

Treat the ten type names as exact strings: `anatomy`, `biological_process`, `cellular_component`, `disease`, `drug`, `effect/phenotype`, `exposure`, `gene/protein`, `molecular_function`, `pathway`. Generate the relation inventory from the acquired TxGNN release; do not import a generic PrimeKG relation list and assume it matches.

Three identifier spaces require explicit crosswalks:

1. External identity: a typed identifier such as `(drug, DB...)`, a MONDO identifier, or an opaque grouped-disease string.
2. Source global index: `node_index`, `x_index`, `y_index` in the released graph files.
3. Model-local index: `x_idx`, `y_idx`, contiguous separately within each node type.

Create a registry with `node_type, local_idx, raw_external_id, reference_id, node_name, node_source`; keep global-index and disease-membership tables separately where the relationship is one-to-many. Names and `display_relation` are descriptive metadata, not keys. An original MONDO concept may map to a grouped node. Preserve the group's exact string and membership; ungrouped diseases need an identity fallback rather than a fabricated group.

The public compatibility normalizer turns numeric-looking IDs into a float string unless they contain underscores. For example, `00123` becomes `123.0`; grouped strings survive intact. Retain the raw ID, audit collisions, and store the reference form in a separate column. Per-type indices follow lexicographically sorted unique reference strings, so `10.0` sorts before `2.0`.

### Independent preprocessing tasks, in order

| Task | Required output and check |
|---|---|
| Read immutable raw tables | Retain source-row order and a stable source-row identifier; check nulls, type/source conflicts and headers |
| Profile canonical relations | Count each `(source_type, relation, target_type)`; distinguish observed absence relations from generated negative examples |
| Reproduce representative selection | Audit the reference's homogeneous-pair deduplication and first-observed-type orientation; preserve which source rows survive |
| Normalize identities | Save lossless raw-to-reference mappings and report every normalization collision |
| Assign local indices | Independently construct per-type lexicographic registries and compare every identity/index with the oracle |
| Create split membership | Save realized masks and row memberships before reverse augmentation |
| Add reverse rows | Keep facts and their reverses in the same partition; retain multiplicity and reference row order |
| Assemble adjacency | Use training rows only, but full processed node counts |

The reference has non-obvious compatibility rules. Homogeneous recognition uses a relation-name convention; duplicate selection occurs before ID normalization. Disease-area preprocessing performs endpoint-only deduplication, omitting relation from its key. Reverse augmentation does not subsequently deduplicate self-loops. These are audit targets: quantify their effect on the acquired data before changing a rule. Any correction belongs in a separately versioned variant.

## 4. Split-aware graph preparation

A training graph cannot be defined before its split/mask is frozen. This belongs in step 1 even though training and evaluation come later.

Keep the node registry derived from the complete processed KG. Construct adjacency from reverse-augmented training rows. A held-out disease may retain legitimate biological neighbors and must retain its input row even when it becomes isolated. Do not rerun the largest-connected-component filter after holding out edges.

For heterogeneous endpoints, reverse relation names use `rev_`; for homogeneous endpoints, the reverse row retains the original relation name. A held-out fact and its reverse must both be absent from training adjacency. Distinct relation types between the same endpoints remain distinct facts unless the explicitly reproduced source rule removes one.

### Disease-area branch

For `autoimmune`, the pinned splitter begins with DOID `417`: ontology descendants → MONDO crosswalk → original/grouped nodes → global indices. The default executable neighborhood call uses **two hops**, despite a nearby one-hop comment. Its default test budget uses 5% of the global edge list; the separate `one_hop=True` path uses a fraction of the local neighborhood. Record the selected branch and actual sampled mask.

The fold seed does not seed area preprocessing. The latter uses ambient NumPy randomness. The notebook's explicit preprocessing seed defines an owned run; it does not recover an unknown historical mask. Freeze the mask as actual global-index pairs. Also save both cached base test rows and the final evaluation view: area filtering removes some reverse therapeutic rows after the base `test.csv` is written.

The independent area splitter must additionally match the pinned PyG neighborhood semantics, sampling population/order, validation sampling rule, mapping files and duplicate handling. No whole-GNN training is needed to check these outputs.

## 5. Capture a reference oracle

The next cell is a guarded adapter for an already working isolated reference environment. It verifies the checked-out SHA, stages fresh data paths, records the area mask when applicable, exports indexed tables, graph edge order, node type/relation order and actual inputs. A new work directory prevents an old cache from silently selecting a different mask.

This is reference instrumentation, not the independent implementation. The later independent builder must consume the same raw assets and frozen split contract, produce its own tables, and compare them with this snapshot. The exporter intentionally stops before constructing or training a GNN.


In [2]:
def capture_reference(raw_dir, repo_dir, work_dir, config):
    import inspect
    import platform
    import random
    import subprocess
    import torch
    import dgl
    from txgnn import TxData
    from txgnn.utils import initialize_node_embedding
    from txgnn.data_splits.datasplit import DataSplitter

    raw_dir, repo_dir, work_dir = map(Path, (raw_dir, repo_dir, work_dir))
    sha = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
    if sha != config['reference_sha']:
        raise ValueError('Reference checkout does not match the configured SHA.')
    subprocess.run(['git', '-C', str(repo_dir), 'diff', '--exit-code', 'HEAD', '--', 'txgnn'], check=True)
    installed_file = Path(inspect.getfile(TxData)).resolve()
    if repo_dir.resolve() not in installed_file.parents:
        raise ValueError('TxData is imported from a different installation: ' + str(installed_file))
    if torch.get_default_dtype() != torch.float32:
        raise ValueError('Reference input recipe requires default float32.')
    lock = json.loads((raw_dir / 'asset_lock.json').read_text())
    for filename in ASSETS:
        if file_sha256(raw_dir / filename) != lock[filename]['sha256']:
            raise ValueError('Raw asset digest mismatch: ' + filename)

    work_dir.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix='reference_', dir=str(work_dir)))
    data_dir, out = run_dir / 'data', run_dir / 'snapshot'
    data_dir.mkdir()
    out.mkdir()
    for filename in ASSETS:
        try:
            os.link(str(raw_dir / filename), str(data_dir / filename))
        except OSError:
            shutil.copy2(str(raw_dir / filename), str(data_dir / filename))
    shutil.copytree(str(repo_dir / 'data' / 'disease_files'), str(data_dir / 'disease_files'))

    def numpy_state():
        name, keys, position, has_gauss, cached_gaussian = np.random.get_state()
        return {'algorithm': name, 'keys': keys.tolist(), 'position': position,
                'has_gauss': has_gauss, 'cached_gaussian': cached_gaussian}

    # Seed after imports. This defines the owned protocol, not an unknown historical run.
    np.random.seed(config['preprocess_seed'])
    random.seed(config['preprocess_seed'])
    states = {'numpy_preparation_before': numpy_state(),
              'python_preparation_before': random.getstate()}
    original_area_selector = DataSplitter.get_test_kg_for_disease

    def record_area_selection(instance, *args, **kwargs):
        selected = original_area_selector(instance, *args, **kwargs)
        selected[['x_index', 'y_index']].drop_duplicates().to_csv(out / 'area_mask_pairs.csv', index=False)
        return selected

    DataSplitter.get_test_kg_for_disease = record_area_selection
    try:
        reference_data = TxData(data_folder_path=str(data_dir))
        reference_data.prepare_split(
            split=config['split'], seed=config['fold_seed'], no_kg=config['no_kg'],
            test_size=config['test_size'], one_hop=config['one_hop'], mask_ratio=config['mask_ratio'],
        )
    finally:
        DataSplitter.get_test_kg_for_disease = original_area_selector

    states['numpy_preparation_after'] = numpy_state()
    states['python_preparation_after'] = random.getstate()
    (out / 'preparation_rng_states.json').write_text(json.dumps(states) + '\n')
    for filename, frame in [
        ('processed.csv', reference_data.df), ('train.csv', reference_data.df_train),
        ('valid.csv', reference_data.df_valid), ('evaluation_test.csv', reference_data.df_test),
    ]:
        frame.to_csv(out / filename, index=False)
    cached_test_files = list(data_dir.rglob('test.csv'))
    if len(cached_test_files) != 1:
        raise ValueError('Expected exactly one base test cache in the fresh reference directory.')
    shutil.copy2(str(cached_test_files[0]), str(out / 'base_test.csv'))

    graph = reference_data.G.to('cpu')
    torch.manual_seed(config['input_seed'])
    np.save(out / 'torch_cpu_rng_before_inputs.npy', torch.get_rng_state().cpu().numpy(), allow_pickle=False)
    graph = initialize_node_embedding(graph, config['input_dim'])
    np.save(out / 'torch_cpu_rng_after_inputs.npy', torch.get_rng_state().cpu().numpy(), allow_pickle=False)
    node_types = list(graph.ntypes)
    canonical_etypes = [tuple(t) for t in graph.canonical_etypes]
    schema = {
        'node_types': node_types,
        'node_counts': {t: int(graph.number_of_nodes(t)) for t in node_types},
        'canonical_etypes': [list(t) for t in canonical_etypes],
        'edge_type_name_order': list(graph.etypes),
        'relation_parameter_rows': [{'canonical_etype': list(t), 'row': i} for i, t in enumerate(canonical_etypes)],
        'reverse_relations': {},
    }
    validation_etypes = list(canonical_etypes)
    base_types = reference_data.df[['x_type', 'relation', 'y_type']].drop_duplicates()
    for source_type, relation, target_type in base_types.itertuples(index=False, name=None):
        forward = (source_type, relation, target_type)
        inverse_name = relation if source_type == target_type else 'rev_' + relation
        backward = (target_type, inverse_name, source_type)
        for triple in [forward, backward]:
            if triple not in validation_etypes:
                validation_etypes.append(triple)
        if source_type != target_type:
            schema['reverse_relations'][inverse_name] = relation
    schema['validation_canonical_etypes'] = [list(t) for t in validation_etypes]
    for i, node_type in enumerate(node_types):
        values = graph.nodes[node_type].data['inp']
        if values.requires_grad or values.dtype != torch.float32:
            raise ValueError('Unexpected reference node-input contract.')
        np.save(out / ('input_{:03d}.npy'.format(i)), values.detach().cpu().numpy(), allow_pickle=False)
    for i, canonical_etype in enumerate(canonical_etypes):
        source, target = graph.edges(etype=canonical_etype, order='eid')
        array = np.stack([source.cpu().numpy(), target.cpu().numpy()]).astype(np.int64, copy=False)
        np.save(out / ('edge_{:03d}.npy'.format(i)), array, allow_pickle=False)
    (out / 'schema.json').write_text(json.dumps(schema, indent=2) + '\n')
    environment = {'python': sys.version, 'platform': platform.platform(),
                   'numpy': np.__version__, 'pandas': pd.__version__,
                   'torch': torch.__version__, 'dgl': dgl.__version__}
    (out / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
    freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    (out / 'requirements_observed.txt').write_text(freeze)
    auxiliary_files = [repo_dir / 'txgnn' / 'data_splits' / name for name in
                       ['HumanDO.obo', 'mondo_references.csv', 'kg_grouped_diseases_bert_map.csv']]
    auxiliary_files.extend(sorted((repo_dir / 'data' / 'disease_files').glob('*.csv')))
    auxiliary_hashes = {str(p.relative_to(repo_dir)): file_sha256(p) for p in auxiliary_files}
    manifest = {'schema_version': 1, 'config': config, 'assets': lock,
                'auxiliary_sha256': auxiliary_hashes,
                'status': 'reference snapshot; independent full-data parity pending',
                'files': {p.name: file_sha256(p) for p in sorted(out.iterdir()) if p.is_file()}}
    (out / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
    return out

if RUN_REFERENCE:
    REFERENCE_SNAPSHOT = capture_reference(WORK / 'raw', PINNED_REPO, WORK, CONFIG)
    print('Reference snapshot:', REFERENCE_SNAPSHOT)
else:
    print('Reference exporter ready; legacy runtime and full-data execution are pending.')


Reference exporter ready; legacy runtime and full-data execution are pending.


## 6. Independently assemble a typed graph

The following constructor is an executable building block. Its inputs are already-normalized, indexed frames, explicit node-type and canonical-relation orders, and a reverse mapping. It preserves zero-degree nodes, parallel relation types, edge order and multiplicity. It validates identifier/index mappings instead of inferring identities from train-only endpoints.

For the first integration check, feed the oracle's indexed tables to this independently written constructor and compare its arrays with DGL's. Next, replace those inputs with independently prepared tables derived from the frozen raw assets and masks. Only the second comparison verifies independent data preparation; the first isolates graph-adapter correctness.

A small synthetic fixture is intentionally separate from the real dataset. It checks overlapping bare IDs across types, an isolated node, parallel relations, repeated self-loops, a reverse-edge holdout leak, malformed indices, invented source facts, holdout-only relation types, and ordered versus multiset hashing. Passing these checks is useful evidence about the constructor, not a substitute for full-data parity.


In [3]:
"""Owned, backend-neutral graph assembly from already prepared TxGNN tables.

This does NOT harmonize raw IDs, group diseases, or create experimental folds.
IDs must already be normalized strings; local indices may be integral floats.
The manifest supplies ordered node_types, ordered training canonical_etypes,
optional validation_canonical_etypes for the complete schema, and a
reverse_relations mapping from reverse relation name to forward relation name.
Memory use includes pandas copies and joins; benchmark before a full-scale run.
"""
from dataclasses import dataclass
import hashlib
import json
from itertools import combinations
import numpy as np
import pandas as pd

COLS = ["x_type", "x_id", "x_idx", "relation", "y_type", "y_id", "y_idx"]
FACT = ["x_type", "x_id", "relation", "y_type", "y_id"]
TRIPLE = ["x_type", "relation", "y_type"]


def _table(frame, label):
    missing = set(COLS) - set(frame.columns)
    if missing:
        raise ValueError(f"{label}: missing columns {sorted(missing)}")
    out = frame[COLS].copy()
    for col in FACT:
        if not out[col].map(lambda v: isinstance(v, str) and bool(v)).all():
            raise ValueError(f"{label}.{col}: nonempty strings required")
    for col in ["x_idx", "y_idx"]:
        values = out[col].to_numpy()
        if (not pd.api.types.is_numeric_dtype(out[col].dtype)
                or pd.api.types.is_bool_dtype(out[col].dtype)):
            raise ValueError(f"{label}.{col}: numeric integer indices required")
        if (not np.isfinite(values).all() or (values < 0).any()
                or (values >= 2**63).any()
                or (values != np.floor(values)).any()):
            raise ValueError(f"{label}.{col}: invalid local index")
        out[col] = values.astype(np.int64)
    return out


def _registry(full, node_types):
    pieces = []
    for side in ["x", "y"]:
        part = full[[f"{side}_type", f"{side}_id", f"{side}_idx"]].copy()
        part.columns = ["type", "id", "idx"]
        pieces.append(part)
    nodes = pd.concat(pieces, ignore_index=True).drop_duplicates()
    if set(nodes.type) != set(node_types):
        raise ValueError("Manifest node types must equal the full-table inventory")
    registry = {}
    for ntype in node_types:
        part = nodes[nodes.type == ntype].sort_values("idx")
        if part.id.duplicated().any() or part.idx.duplicated().any():
            raise ValueError(f"{ntype}: ID/index mapping is not a bijection")
        if not np.array_equal(part.idx.to_numpy(), np.arange(len(part))):
            raise ValueError(f"{ntype}: local indices are not contiguous from zero")
        registry[ntype] = tuple(part.id)
    return registry


def _canonical_facts(frame, reverse_relations):
    out = frame[FACT].copy()
    rev = out.relation.isin(reverse_relations)
    for left, right in [("x_type", "y_type"), ("x_id", "y_id")]:
        out.loc[rev, [left, right]] = out.loc[rev, [right, left]].to_numpy()
    out.loc[rev, "relation"] = out.loc[rev, "relation"].map(reverse_relations)
    swap = (out.x_type == out.y_type) & (out.x_id > out.y_id)
    out.loc[swap, ["x_id", "y_id"]] = out.loc[swap, ["y_id", "x_id"]].to_numpy()
    return out.drop_duplicates()


def assert_no_leakage(splits, reverse_relations, full_processed=None):
    """Reject shared typed facts, including their reverse equivalents.

    Same-type relations are treated as undirected, matching this TxGNN target.
    When full_processed is supplied, also reject facts absent from that source.
    This does not certify the intended disease-split membership or proportions.
    """
    facts = {name: _canonical_facts(frame, reverse_relations)
             for name, frame in splits.items()}
    if full_processed is not None:
        source = _canonical_facts(full_processed, reverse_relations)
        for name, frame in facts.items():
            checked = frame.merge(source, on=FACT, how="left", indicator=True)
            missing = checked[checked["_merge"] == "left_only"]
            if len(missing):
                example = missing.iloc[0][FACT].to_dict()
                raise ValueError(f"{name}: fact absent from full source: {example}")
    for a, b in combinations(facts, 2):
        overlap = facts[a].merge(facts[b], on=FACT, how="inner")
        if len(overlap):
            example = overlap.iloc[0].to_dict()
            raise ValueError(f"Leakage between {a} and {b}: {example}")


@dataclass
class OwnedGraph:
    node_types: tuple
    canonical_etypes: tuple
    node_ids: dict
    edges: dict

    @property
    def num_nodes(self):
        return {kind: len(ids) for kind, ids in self.node_ids.items()}

    def logical_hash(self, *, ordered=True):
        """SHA-256 of schema, node identities, and int64 COO edge content.

        ordered=False ignores within-relation edge order only. It retains
        duplicate multiplicity, node row order, and relation parameter order.
        """
        digest = hashlib.sha256()

        def record(value):
            payload = json.dumps(value, ensure_ascii=False,
                                 separators=(",", ":")).encode("utf-8")
            digest.update(len(payload).to_bytes(8, "little"))
            digest.update(payload)

        record(["owned-graph-v1", self.node_types, self.canonical_etypes])
        for ntype in self.node_types:
            record([ntype, self.node_ids[ntype]])
        for etype in self.canonical_etypes:
            pairs = self.edges[etype].T
            if not ordered and len(pairs):
                pairs = pairs[np.lexsort((pairs[:, 1], pairs[:, 0]))]
            record([etype, len(pairs)])
            digest.update(np.ascontiguousarray(pairs, dtype="<i8").tobytes())
        return digest.hexdigest()


def build_graph(full_processed, splits, manifest):
    """Build training COO arrays; validate held-out tables without adding edges.

    splits must contain train/valid/test. Empty frames need the same columns
    with numeric index dtypes. Validation-only relations do not create training
    arrays. Pin canonical_etypes to the actual reference training schema.
    """
    if set(splits) != {"train", "valid", "test"}:
        raise ValueError("Exactly train, valid, and test tables are required")
    node_types = tuple(manifest["node_types"])
    etypes = tuple(tuple(e) for e in manifest["canonical_etypes"])
    validation_etypes = tuple(tuple(e) for e in manifest.get(
        "validation_canonical_etypes", etypes))
    reverse = dict(manifest["reverse_relations"])
    if (len(set(node_types)) != len(node_types)
            or not all(isinstance(t, str) and t for t in node_types)
            or len(set(etypes)) != len(etypes)
            or len(set(validation_etypes)) != len(validation_etypes)):
        raise ValueError("Manifest type orders contain duplicates or invalid names")
    if not set(etypes) <= set(validation_etypes):
        raise ValueError("Training schema is absent from the complete validation schema")
    relation_types = {}
    for etype in validation_etypes:
        if (len(etype) != 3 or not all(isinstance(t, str) and t for t in etype)
                or etype[0] not in node_types or etype[2] not in node_types):
            raise ValueError(f"Invalid canonical edge type {etype}")
        src, relation, dst = etype
        if relation in relation_types:
            raise ValueError(f"Ambiguous relation name: {relation}")
        relation_types[relation] = (src, dst)
    for backward, forward in reverse.items():
        if (backward == forward or forward in reverse
                or backward not in relation_types or forward not in relation_types
                or relation_types[backward] != relation_types[forward][::-1]):
            raise ValueError(f"Invalid reverse relation mapping: {backward}")
    if any(r.startswith("rev_") and r not in reverse for r in relation_types):
        raise ValueError("All rev_ relation names require explicit reverse mappings")
    full = _table(full_processed, "full")
    prepared = {name: _table(frame, name) for name, frame in splits.items()}
    registry = _registry(full, node_types)
    lookup = {kind: {external: i for i, external in enumerate(ids)}
              for kind, ids in registry.items()}
    for name, frame in {"full": full, **prepared}.items():
        allowed = set(etypes if name == "train" else validation_etypes)
        observed = set(frame[TRIPLE].itertuples(index=False, name=None))
        if not observed <= allowed:
            raise ValueError(f"{name}: unknown canonical edge types {observed - allowed}")
        for side in ["x", "y"]:
            for ntype, part in frame.groupby(f"{side}_type", sort=False):
                expected = part[f"{side}_id"].map(lookup[ntype])
                if expected.isna().any() or not np.array_equal(
                        expected.to_numpy(), part[f"{side}_idx"].to_numpy()):
                    raise ValueError(f"{name}: invalid {side} endpoint identity/index")
    assert_no_leakage(prepared, reverse, full_processed=full)
    train = prepared["train"]
    edges = {}
    for src, relation, dst in etypes:
        mask = ((train.x_type == src) & (train.relation == relation)
                & (train.y_type == dst))
        coo = train.loc[mask, ["x_idx", "y_idx"]].to_numpy(dtype=np.int64).T.copy()
        coo.setflags(write=False)
        edges[(src, relation, dst)] = coo
    return OwnedGraph(node_types, etypes, registry, edges)


In [4]:
"""Small structural tests; these are not tests against the released full KG."""
import copy
import unittest
import numpy as np
import pandas as pd


def table(rows):
    frame = pd.DataFrame(rows, columns=COLS)
    return frame.astype({"x_idx": "int64", "y_idx": "int64"})


def fixture():
    # Bare ID "shared" belongs to both a drug and a disease. Disease d2 is
    # present in the full KG but isolated in the training graph.
    forward = [
        ("drug", "shared", 0, "indication", "disease", "shared", 0),
        ("drug", "drug1", 1, "indication", "disease", "d1", 1),
        ("drug", "shared", 0, "contraindication", "disease", "shared", 0),
        ("disease", "shared", 0, "disease_disease", "disease", "shared", 0),
    ]
    heldout = ("drug", "drug1", 1, "indication", "disease", "d2", 2)
    rows = forward + [
        ("disease", "shared", 0, "rev_indication", "drug", "shared", 0),
        ("disease", "d1", 1, "rev_indication", "drug", "drug1", 1),
        ("disease", "shared", 0, "rev_contraindication", "drug", "shared", 0),
        forward[-1],  # Homogeneous self-loop duplicate after reversal.
    ]
    full = table(forward + [heldout])
    splits = {"train": table(rows), "valid": table([heldout]), "test": table([])}
    manifest = {
        "node_types": ["drug", "disease"],
        "canonical_etypes": [
            ["drug", "indication", "disease"],
            ["disease", "rev_indication", "drug"],
            ["drug", "contraindication", "disease"],
            ["disease", "rev_contraindication", "drug"],
            ["disease", "disease_disease", "disease"],
        ],
        "reverse_relations": {
            "rev_indication": "indication",
            "rev_contraindication": "contraindication",
        },
    }
    return full, splits, manifest


class GraphTests(unittest.TestCase):
    def test_inventory_identity_parallel_edges_and_multiplicity(self):
        full, splits, manifest = fixture()
        graph = build_graph(full, splits, manifest)
        self.assertEqual(graph.num_nodes, {"drug": 2, "disease": 3})
        self.assertEqual(graph.node_ids["drug"][0], graph.node_ids["disease"][0])
        self.assertEqual(graph.node_types, ("drug", "disease"))
        self.assertEqual(graph.canonical_etypes, tuple(map(tuple, manifest["canonical_etypes"])))
        indication = graph.edges[("drug", "indication", "disease")]
        np.testing.assert_array_equal(indication, [[0, 1], [0, 1]])
        self.assertEqual(graph.edges[("drug", "contraindication", "disease")].shape[1], 1)
        self.assertEqual(graph.edges[("disease", "disease_disease", "disease")].shape[1], 2)
        self.assertFalse(indication.flags.writeable)
        self.assertFalse(any(2 in edges for edges in graph.edges.values()))

    def test_reverse_heldout_leak_rejected(self):
        full, splits, manifest = fixture()
        leak = table([("disease", "shared", 0, "rev_indication", "drug", "shared", 0)])
        splits["test"] = leak
        with self.assertRaisesRegex(ValueError, "Leakage"):
            build_graph(full, splits, manifest)

    def test_homogeneous_reverse_leak_rejected(self):
        full, splits, manifest = fixture()
        first = table([("disease", "shared", 0, "disease_disease", "disease", "d1", 1)])
        reverse = table([("disease", "d1", 1, "disease_disease", "disease", "shared", 0)])
        full = pd.concat([full, first], ignore_index=True)
        splits["train"] = pd.concat([splits["train"], first], ignore_index=True)
        splits["test"] = reverse
        with self.assertRaisesRegex(ValueError, "Leakage"):
            build_graph(full, splits, manifest)

    def test_bad_local_index_rejected(self):
        for value in [-1.0, 0.5, float("nan"), 100.0]:
            full, splits, manifest = fixture()
            splits["train"]["x_idx"] = splits["train"].x_idx.astype(float)
            splits["train"].loc[0, "x_idx"] = value
            with self.subTest(value=value), self.assertRaises(ValueError):
                build_graph(full, splits, manifest)

    def test_registry_bijection_and_contiguity(self):
        full, splits, manifest = fixture()
        broken = full.copy()
        broken.loc[1, "y_idx"] = 0
        with self.assertRaisesRegex(ValueError, "bijection"):
            build_graph(broken, splits, manifest)
        broken = full.copy()
        broken.loc[broken.y_id == "d2", "y_idx"] = 4
        with self.assertRaisesRegex(ValueError, "contiguous"):
            build_graph(broken, splits, manifest)

    def test_ordered_and_multiset_hashes(self):
        full, splits, manifest = fixture()
        a = build_graph(full, splits, manifest)
        moved = dict(splits)
        moved["train"] = splits["train"].iloc[::-1].reset_index(drop=True)
        b = build_graph(full, moved, manifest)
        self.assertNotEqual(a.logical_hash(), b.logical_hash())
        self.assertEqual(a.logical_hash(ordered=False), b.logical_hash(ordered=False))
        moved["train"] = splits["train"].iloc[:-1]
        c = build_graph(full, moved, manifest)
        self.assertNotEqual(a.logical_hash(ordered=False), c.logical_hash(ordered=False))

    def test_relation_name_ambiguity_rejected(self):
        full, splits, manifest = fixture()
        changed = copy.deepcopy(manifest)
        changed["canonical_etypes"].append(["disease", "indication", "drug"])
        with self.assertRaisesRegex(ValueError, "Ambiguous relation"):
            build_graph(full, splits, changed)

    def test_invented_source_fact_rejected(self):
        full, splits, manifest = fixture()
        # Both endpoints and the relation are valid, but the fact is absent.
        splits["test"] = table([
            ("drug", "shared", 0, "indication", "disease", "d2", 2)])
        with self.assertRaisesRegex(ValueError, "absent from full source"):
            build_graph(full, splits, manifest)

    def test_holdout_only_relation_does_not_expand_training_schema(self):
        full, splits, manifest = fixture()
        heldout = ("drug", "shared", 0, "off-label use", "disease", "d2", 2)
        reverse = ("disease", "d2", 2, "rev_off-label use", "drug", "shared", 0)
        full = pd.concat([full, table([heldout])], ignore_index=True)
        splits["test"] = table([heldout, reverse])
        manifest["validation_canonical_etypes"] = manifest["canonical_etypes"] + [
            ["drug", "off-label use", "disease"],
            ["disease", "rev_off-label use", "drug"],
        ]
        manifest["reverse_relations"]["rev_off-label use"] = "off-label use"
        graph = build_graph(full, splits, manifest)
        self.assertEqual(graph.canonical_etypes,
                         tuple(map(tuple, manifest["canonical_etypes"])))
        self.assertNotIn(("drug", "off-label use", "disease"), graph.edges)
        self.assertNotIn(("disease", "rev_off-label use", "drug"), graph.edges)
        # A complete validation schema must not admit these into training.
        splits["train"] = pd.concat([splits["train"], table([heldout])], ignore_index=True)
        with self.assertRaisesRegex(ValueError, "train: unknown canonical"):
            build_graph(full, splits, manifest)


def run_toy_checks():
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(GraphTests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError("Owned graph toy checks failed")
    return {"tests_run": result.testsRun, "failures": len(result.failures),
            "errors": len(result.errors)}



print(run_toy_checks())


test_bad_local_index_rejected (__main__.GraphTests.test_bad_local_index_rejected) ... ok
test_holdout_only_relation_does_not_expand_training_schema (__main__.GraphTests.test_holdout_only_relation_does_not_expand_training_schema) ... ok
test_homogeneous_reverse_leak_rejected (__main__.GraphTests.test_homogeneous_reverse_leak_rejected) ... ok
test_invented_source_fact_rejected (__main__.GraphTests.test_invented_source_fact_rejected) ... ok
test_inventory_identity_parallel_edges_and_multiplicity (__main__.GraphTests.test_inventory_identity_parallel_edges_and_multiplicity) ... ok
test_ordered_and_multiset_hashes (__main__.GraphTests.test_ordered_and_multiset_hashes) ... ok
test_registry_bijection_and_contiguity (__main__.GraphTests.test_registry_bijection_and_contiguity) ... ok
test_relation_name_ambiguity_rejected (__main__.GraphTests.test_relation_name_ambiguity_rejected) ... ok
test_reverse_heldout_leak_rejected (__main__.GraphTests.test_reverse_heldout_leak_rejected) ... ok

----------

## 7. Initialize and preserve node inputs

The reference creates a whole matrix of shape `[N_type, input_dim]` for each type, in the realized graph's node-type order, and applies PyTorch Xavier uniform with default gain. It freezes the result. These are neither learned embedding tables nor ClinicalBERT disease-name vectors.

For the complete two-dimensional matrix, the bound is `sqrt(6 / (N_type + input_dim))`. A row-by-row Xavier call changes the distribution. A different random-number implementation can match that distribution while producing different values. For input parity, use the pinned CPU torch initializer or directly load the actual reference-exported matrices.

The independent implementation below uses the same mathematical initialization primitive and checks it against the exported reference. It restores the caller's CPU Torch RNG after the check. Store the before/after states separately if later parameter initialization must continue the same sequence.

Input dimension 100 follows `reproduce/train.py`. The attached preprint's page 36 gives 512; that should be a separately named experiment. Do not silently mix the two configurations.


In [5]:
def make_owned_inputs(node_counts, node_type_order, input_dim, input_seed):
    import torch
    caller_state = torch.get_rng_state().clone()
    try:
        torch.manual_seed(input_seed)
        before = torch.get_rng_state().clone()
        matrices = {}
        for node_type in node_type_order:
            values = torch.empty((node_counts[node_type], input_dim), dtype=torch.float32, device='cpu')
            torch.nn.init.xavier_uniform_(values)
            values.requires_grad_(False)
            matrices[node_type] = values
        after = torch.get_rng_state().clone()
    finally:
        torch.set_rng_state(caller_state)
    return matrices, before, after


### Run the real adapter/input comparison when the reference snapshot exists

This comparison uses reference-prepared indexed tables to isolate independent graph construction and initialization. It cannot establish that independent raw-data preparation is complete. After implementing that upstream work package, replace `processed.csv` and split frames below with independently generated tables and compare node registries and split memberships as well.

The base test table is used for disjointness and source-membership checks. The separately exported `evaluation_test.csv` remains the exact evaluation view. For a deliberately overlapping setting such as `full_graph` (where the reference reuses validation as test), supply a separately documented contract rather than weakening these held-out gates.


In [6]:
def compare_real_snapshot(snapshot_dir, output_parent):
    import subprocess
    snapshot_dir = Path(snapshot_dir)
    source_manifest = json.loads((snapshot_dir / 'manifest.json').read_text())
    for filename, expected in source_manifest['files'].items():
        if file_sha256(snapshot_dir / filename) != expected:
            raise ValueError('Reference snapshot changed: ' + filename)
    schema = json.loads((snapshot_dir / 'schema.json').read_text())
    config = source_manifest['config']

    def read_frame(filename):
        return pd.read_csv(snapshot_dir / filename,
                           dtype={'x_id': str, 'y_id': str, 'x_type': str,
                                  'y_type': str, 'relation': str}, low_memory=False)

    full = read_frame('processed.csv')
    splits = {'train': read_frame('train.csv'), 'valid': read_frame('valid.csv'),
              'test': read_frame('base_test.csv')}
    owned = build_graph(full, splits, schema)
    if owned.num_nodes != schema['node_counts']:
        raise AssertionError('Node inventory differs from the reference graph.')
    for i, etype in enumerate(owned.canonical_etypes):
        expected = np.load(snapshot_dir / ('edge_{:03d}.npy'.format(i)), allow_pickle=False)
        np.testing.assert_array_equal(owned.edges[etype], expected)

    inputs, before, after = make_owned_inputs(owned.num_nodes, owned.node_types,
                                            config['input_dim'], config['input_seed'])
    np.testing.assert_array_equal(before.cpu().numpy(), np.load(snapshot_dir / 'torch_cpu_rng_before_inputs.npy', allow_pickle=False))
    np.testing.assert_array_equal(after.cpu().numpy(), np.load(snapshot_dir / 'torch_cpu_rng_after_inputs.npy', allow_pickle=False))
    output_parent = Path(output_parent)
    output_parent.mkdir(parents=True, exist_ok=True)
    out = Path(tempfile.mkdtemp(prefix='adapter_check_', dir=str(output_parent)))
    input_digests = {}
    for i, ntype in enumerate(owned.node_types):
        values = inputs[ntype].detach().cpu().numpy()
        expected = np.load(snapshot_dir / ('input_{:03d}.npy'.format(i)), allow_pickle=False)
        np.testing.assert_array_equal(values, expected)
        if values.dtype != np.float32 or not np.isfinite(values).all():
            raise AssertionError('Invalid input matrix: ' + ntype)
        header = json.dumps([ntype, list(values.shape), 'float32'], separators=(',', ':')).encode()
        h = hashlib.sha256(len(header).to_bytes(8, 'little') + header)
        h.update(np.ascontiguousarray(values, dtype='<f4').tobytes())
        input_digests[ntype] = h.hexdigest()
        np.save(out / ('input_{:03d}.npy'.format(i)), values, allow_pickle=False)
    for i, etype in enumerate(owned.canonical_etypes):
        np.save(out / ('edge_{:03d}.npy'.format(i)), owned.edges[etype], allow_pickle=False)
    (out / 'node_reference_ids.json').write_text(json.dumps(owned.node_ids) + '\n')
    (out / 'schema.json').write_text(json.dumps(schema, indent=2) + '\n')
    report = {
        'scope': 'reference-prepared tables to independently built graph and input matrices',
        'full_owned_preparation_complete': False,
        'independent_raw_preparation_parity': 'pending',
        'ordered_graph_arrays_equal': True,
        'frozen_inputs_equal': True,
        'graph_ordered_hash': owned.logical_hash(),
        'graph_edge_multiset_hash': owned.logical_hash(ordered=False),
        'input_logical_hashes': input_digests,
        'input_dim': config['input_dim'],
        'artifact_sha256': {p.name: file_sha256(p) for p in sorted(out.iterdir()) if p.is_file()},
    }
    (out / 'checks.json').write_text(json.dumps(report, indent=2) + '\n')

    # A new process reopens the stored arrays and schema and verifies their content.
    verifier = """
from pathlib import Path
import hashlib, json, sys
import numpy as np
p = Path(sys.argv[1])
c = json.loads((p / 'checks.json').read_text())
s = json.loads((p / 'schema.json').read_text())
for name, wanted in c['artifact_sha256'].items():
    h = hashlib.sha256()
    with (p / name).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    assert h.hexdigest() == wanted, name
for i, ntype in enumerate(s['node_types']):
    x = np.load(p / ('input_{:03d}.npy'.format(i)), allow_pickle=False)
    assert x.shape == (s['node_counts'][ntype], c['input_dim']) and x.dtype == np.float32
for i, (src, rel, dst) in enumerate(s['canonical_etypes']):
    e = np.load(p / ('edge_{:03d}.npy'.format(i)), allow_pickle=False)
    assert e.ndim == 2 and e.shape[0] == 2 and e.dtype == np.int64
    if e.shape[1]:
        assert e.min() >= 0 and e[0].max() < s['node_counts'][src] and e[1].max() < s['node_counts'][dst]
print('Fresh-process array and schema verification passed.')
"""
    subprocess.run([sys.executable, '-c', verifier, str(out)], check=True)
    report['fresh_process_array_and_schema_verification'] = True
    (out / 'checks.json').write_text(json.dumps(report, indent=2) + '\n')
    return out, report

if RUN_REAL_COMPARISON:
    if REFERENCE_SNAPSHOT is None:
        raise ValueError('Export a reference snapshot or set its path first.')
    comparison_dir, comparison_report = compare_real_snapshot(REFERENCE_SNAPSHOT, WORK)
    print(json.dumps(comparison_report, indent=2))
else:
    print('Real graph and input parity checks are ready; not executed.')


Real graph and input parity checks are ready; not executed.


## 8. Final owned bundle and model-facing interface

Use ordinal filenames because literal type names include `/`. Keep the full semantic type names in the schema.

| Artifact | Contents |
|---|---|
| `manifest.json` | Preparation version, reference SHA, source/config/environment hashes, artifact inventory and status |
| `nodes.parquet` | Typed local indices, raw/reference identifiers, names and provenance |
| `source_node_crosswalk.parquet` | Source global indices linked to model nodes |
| `disease_membership.parquet` | Original MONDO members, released groups, ungrouped identity mappings |
| `relations.json` | Ordered canonical triples, inverse mapping and future decoder-row mapping |
| `base_edges.parquet` | Representative facts and source-row provenance |
| `split_membership.parquet` | Actual base-fact membership and paired reverse identity |
| `evaluation_targets.parquet` | Exact evaluation view, including area-specific filtering |
| `graph/edge_000.npy`, etc. | Ordered per-relation integer COO arrays |
| `inputs/input_000.npy`, etc. | Actual frozen float32 matrices |
| `rng_states/` | Preparation and input initialization state |
| `checks.json` | Counts, logical hashes, leakage results and reference comparison |

The reference exporter above is an oracle snapshot, not this finished owned bundle: enrich the independent implementation with the raw-ID registry, complete aliases/provenance and split membership before declaring the stage complete.

A future model loader should return `node_counts`, `node_types`, `canonical_etypes`, `edge_index`, `node_inputs`, `node_registry`, `relation_to_parameter_row`, `split_views` and a bundle fingerprint. Loading must not regroup diseases, resample masks, rebuild indices or regenerate inputs.

Hash logical arrays using documented dtype, shape and canonical byte order, alongside file-integrity hashes. Maintain an edge-multiset digest for semantic comparison and an ordered digest for replay-oriented comparison. An opaque DGL serialization hash alone cannot explain whether two graphs represent the same model inputs.

## 9. Acceptance checklist

1. **Asset identity:** fixed file IDs, actual file digests, auxiliary files and reference SHA are recorded; all input headers validate.
2. **Node identity:** every typed ID/index mapping is reversible; indices are contiguous per type; raw-normalization collisions are audited; grouped disease membership resolves consistently.
3. **Graph parity:** exact typed node inventory and relation schema match; per-relation edge multisets match after any explicit remapping; ordered arrays also match for replay-oriented work.
4. **Isolation:** train adjacency uses training membership only; no held-out fact or generated reverse leaks into it. Whole-disease holdouts additionally have no forbidden therapeutic training edge.
5. **Inputs:** correct shapes/dtype, finite values, frozen gradient status, exact row meanings, saved values and logical hashes; equality with the pinned-runtime input oracle.
6. **Reload:** a new process reloads the complete bundle and recovers identical graph/input hashes without random initialization or data preparation.

Use the base split table for paired-reverse checks. The final disease-area evaluation view can be asymmetric by pinned-reference design. If an upstream behavior conflicts with a desired scientific invariant, record the conflict and version the correction explicitly; do not mark both exact parity and a silent correction as passed.

## 10. Execution order and effort

| Work package | Indicative engineering effort | Exit condition |
|---|---:|---|
| Acquire and profile | 1–2 days | Immutable inputs, schema/count report, identifier/group registry design |
| Independent preparation and adjacency | 2–4 days | Independently generated identities, masks, split tables and graphs match the oracle |
| Inputs, export and reload | 1–2 days | Actual tensors saved, full bundle fingerprinted, fresh-process reload passes |

These are planning estimates for an experienced implementer with available assets and a working reference environment, not measured runtimes. Dependency repair or disease-area differences can extend them. This stage is CPU/data engineering work; a training GPU is not needed for its acceptance.

Start with one frozen split and one owned initialization. Expand to multiple splits only after the first bundle passes. Keep graph identity, split identity and input identity separate so a later architectural experiment can reuse the same inputs without silently changing the experiment.

## Primary sources

- User-provided medRxiv v2 paper, pp. 22–23, 35–36: https://www.medrxiv.org/content/10.1101/2023.03.19.23287458v2
- Pinned source root: https://github.com/mims-harvard/TxGNN/tree/0e0203c3939f7d12da27aaf10e55daaff5424023
- Loader and split orchestration: https://github.com/mims-harvard/TxGNN/blob/0e0203c3939f7d12da27aaf10e55daaff5424023/txgnn/TxData.py
- Preparation, reverse edges and inputs: https://github.com/mims-harvard/TxGNN/blob/0e0203c3939f7d12da27aaf10e55daaff5424023/txgnn/utils.py
- Area mappings and masking: https://github.com/mims-harvard/TxGNN/blob/0e0203c3939f7d12da27aaf10e55daaff5424023/txgnn/data_splits/datasplit.py
- Reproduction configuration: https://github.com/mims-harvard/TxGNN/blob/0e0203c3939f7d12da27aaf10e55daaff5424023/reproduce/train.py
- Model initialization orchestration: https://github.com/mims-harvard/TxGNN/blob/0e0203c3939f7d12da27aaf10e55daaff5424023/txgnn/TxGNN.py
- Relation parameter order: https://github.com/mims-harvard/TxGNN/blob/0e0203c3939f7d12da27aaf10e55daaff5424023/txgnn/model.py
- Dataset landing DOI: https://doi.org/10.7910/DVN/IXA7BM
- Upstream schema construction context: https://github.com/mims-harvard/PrimeKG/blob/main/knowledge_graph/build_graph.ipynb
- Xavier initialization definition: https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.xavier_uniform_
